In [0]:
%run ./00-ddl

In [0]:
%run ./01-imports

In [0]:
import os
import requests
import json
import time
import pyspark.sql.functions as F
from datetime import datetime

In [0]:
# Variables
CHECKPOINT_LOCATION_BRONZE = f'{CHECKPOINT_BASE}/raw_data_checkpoints'

# Set Data Location
DATA_LOCATION = f"/Volumes/{CATALOG}/{DATABASE_L}/mlb_gumbo_data"

In [0]:
# Define Bronze Table
current_run = datetime.now()
BRONZE_TABLE = f"{CATALOG}.{DATABASE_B}.raw_data"

# Ingest
query = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("singleVariantColumn", "data")
  .load(f"{DATA_LOCATION}/*.json")
  .withColumn("file_path", F.col("_metadata.file_path"))
  .withColumn("file_name", F.col("_metadata.file_name"))
  .withColumn("file_size", F.col("_metadata.file_size"))
  .withColumn("file_modification_time", F.col("_metadata.file_modification_time"))
  .withColumn("file_batch_time", F.lit(current_run))
  .withColumn("last_update_time", F.current_timestamp())
  .writeStream
  .option("checkpointLocation", f"{CHECKPOINT_LOCATION_BRONZE}")
  .trigger(availableNow=True)
  .toTable(BRONZE_TABLE)
)

query.awaitTermination()